# CSET485 — AI and Society | Assignment 1
**Student roll number:** S24CSEU0728  
**Assigned dataset:** Adult Census Income (last digit 8)  
**Train/test split:** 70% / 30%, `random_state=728`

This notebook is designed to run end-to-end and compute every reported number from the actual UCI Adult dataset. The execution environment used to prepare this file had no network access and no copy of the dataset, so no numerical results are fabricated here. Run all cells in an internet-enabled Python environment; the final cell writes a results-grounded PDF report automatically.

**Dataset source:** UCI Adult dataset (`adult.data`, `adult.test`, `adult.names`).

**Important:** Saved outputs from earlier runs are cleared in this corrected copy. Run all cells from top to bottom to regenerate fresh results.

In [ ]:
# If needed, install dependencies in your environment:
# %pip install pandas numpy scikit-learn statsmodels matplotlib seaborn reportlab

import io, os, urllib.request, warnings, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score
from IPython.display import display
warnings.filterwarnings('ignore')
RANDOM_STATE = 728
OUT_DIR = '.'


In [ ]:
# Load the real UCI data. If the primary URL is unavailable, try the HTTPS mirror.
DATA_URLS = [
    'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data',
    'https://raw.githubusercontent.com/selva86/datasets/master/AdultUCI.csv'
]
cols = ['age','workclass','fnlwgt','education','education_num','marital_status','occupation','relationship','race','sex','capital_gain','capital_loss','hours_per_week','native_country','income']
last_error = None
for url in DATA_URLS:
    try:
        raw = urllib.request.urlopen(url, timeout=30).read()
        if url.endswith('adult.data'):
            df = pd.read_csv(io.BytesIO(raw), header=None, names=cols, skipinitialspace=True, na_values='?')
        else:
            df = pd.read_csv(io.BytesIO(raw))
            # Normalize common alternate column names to the UCI schema if necessary.
            df.columns = [str(c).strip().lower().replace('-', '_').replace(' ', '_') for c in df.columns]
            if 'income' not in df.columns: raise ValueError('Mirror schema does not contain income column')
        break
    except Exception as e:
        last_error = e
else:
    raise RuntimeError('Could not download the real Adult dataset. Check internet access and rerun. Last error: '+repr(last_error))

# UCI adult.data has no header and target labels like <=50K / >50K.
df.columns = [str(c).strip().lower().replace('-', '_').replace(' ', '_') for c in df.columns]
if 'income' not in df.columns:
    raise ValueError(f'Expected income target; got columns: {df.columns.tolist()}')
df = df.replace(r'^\s*\?\s*$', np.nan, regex=True)
df['income_binary'] = df['income'].astype(str).str.strip().str.replace('.', '', regex=False).map({'<=50K':0, '>50K':1})
df = df[df['income_binary'].notna()].copy()
df['income_binary'] = df['income_binary'].astype(int)
print('Dataset shape:', df.shape)
print('Target distribution:')
display(df['income_binary'].value_counts(dropna=False).rename(index={0:'<=50K',1:'>50K'}).to_frame('count'))


## Preprocessing documentation
- **Missing values:** UCI `?` values are converted to missing. Numeric columns are median-imputed; categorical columns are imputed with the most frequent category. Missingness counts are reported below.
- **Categorical encoding:** one-hot encoding with unknown categories ignored, fitted on training data only.
- **Feature scaling:** numeric predictors are standardized in the model pipeline. Scaling is fit on training data only.
- **Outliers:** numeric distributions and extreme values are summarized; values are not automatically deleted because extreme capital gains/losses can be real and removing them could distort the population. Logistic regression's regularization limits coefficient explosion.
- **Leakage prevention:** train/test split occurs before fitted imputers, encoders, and scalers are learned.

In [ ]:
target = 'income_binary'
exclude = ['income', target]
X = df.drop(columns=[c for c in exclude if c in df.columns])
y = df[target]
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

missing_summary = df.isna().sum().sort_values(ascending=False)
print('Missing values per column:')
display(missing_summary[missing_summary > 0].to_frame('missing_count'))
print('Numeric outlier summary (1st/99th percentiles):')
outlier_summary = X[num_cols].quantile([.01,.5,.99]).T
outlier_summary.columns = ['p01','median','p99']
display(outlier_summary)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=True))])
preprocessor = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)], remainder='drop')
Xtr = preprocessor.fit_transform(X_train)
Xte = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()
print('Train rows:', len(X_train), '| Test rows:', len(X_test), '| Encoded features:', len(feature_names))


## Part 1 — Build and compare models
Model A is OLS linear regression for the binary income target (as specified, an OLS probability score; predictions can fall outside [0,1]). Model B is logistic regression. For the comparison, OLS predictions are thresholded at their **test-set median**, and logistic probabilities at **0.5**. The confusion matrix below measures agreement between these two thresholding approaches, not agreement with ground truth.

In [ ]:
ols = LinearRegression().fit(Xtr, y_train)
ols_pred = ols.predict(Xte)
logit = LogisticRegression(max_iter=2000, solver='liblinear', random_state=RANDOM_STATE).fit(Xtr, y_train)
logit_prob = logit.predict_proba(Xte)[:,1]
ols_median = float(np.median(ols_pred))
ols_flag = (ols_pred >= ols_median).astype(int)
logit_flag = (logit_prob >= .5).astype(int)
cm_agreement = confusion_matrix(ols_flag, logit_flag, labels=[0,1])
mismatch = int(np.sum(ols_flag != logit_flag))
mismatch_pct = 100*mismatch/len(y_test)
print(f'OLS test prediction median: {ols_median:.6f}')
print(f'Logistic threshold: 0.500000')
print('Agreement confusion matrix (rows=OLS low/high, columns=logistic low/high):')
display(pd.DataFrame(cm_agreement, index=['OLS low','OLS high'], columns=['Logistic low','Logistic high']))
print(f'Mismatches: {mismatch:,}/{len(y_test):,} ({mismatch_pct:.2f}%)')
print(f'Logistic accuracy vs actual target: {accuracy_score(y_test, logit_flag):.4f}; ROC-AUC: {roc_auc_score(y_test, logit_prob):.4f}')


### Why classifications can differ
OLS minimizes squared prediction error and its median is a data-dependent cutoff; it is not inherently a probability calibration point. Logistic regression estimates a probability through the logistic link and uses a fixed 0.5 cutoff. The two models optimize different loss functions and can rank borderline cases differently. Quote the computed median, confusion matrix, and mismatch percentage in the report.

## Shared helper: statsmodels logistic models
For interpretable coefficients and confidence intervals, use statsmodels on the same one-hot encoded predictors. Sparse matrices are converted to dense arrays here; this dataset is small enough for the expected feature count.

In [ ]:
# Convert encoded matrices into DataFrames with stable feature names.
Xtr_df = pd.DataFrame(Xtr.toarray() if hasattr(Xtr,'toarray') else Xtr, columns=feature_names, index=X_train.index)
Xte_df = pd.DataFrame(Xte.toarray() if hasattr(Xte,'toarray') else Xte, columns=feature_names, index=X_test.index)
# Fit a numerically safer statsmodels logistic model.
# One-hot encoding uses drop='first' to avoid the dummy-variable trap. In subgroup
# fits, some categories may be absent, so remove columns that are constant in that
# particular sample before adding one intercept.
def fit_glm(Xdf, yy):
    xx = Xdf.astype(float).copy()
    varying = xx.nunique(dropna=False) > 1
    xx = xx.loc[:, varying]
    xx = sm.add_constant(xx, has_constant='add')
    model = sm.GLM(np.asarray(yy, dtype=float), xx, family=sm.families.Binomial())
    return model.fit(maxiter=200, tol=1e-8)
full_glm = fit_glm(Xtr_df, y_train)
print(full_glm.summary())


## Part 2.1 — Simpson's paradox / subgroup effect hunt
Use `sex` as the demographic split and compare the coefficient for `education_num` across the full dataset, Female, and Male. If signs do not reverse, the assignment permits reporting a materially stronger/weaker effect with confidence intervals. This is an exploratory subgroup comparison, not proof of causation.

In [ ]:
# Reuse the already encoded training set, but fit subgroup models based on original training rows.
# Determine the encoded feature representing education_num robustly.
key_candidates = [c for c in Xtr_df.columns if c.endswith('__education_num') or c.endswith('education_num')]
key_predictor = key_candidates[0] if key_candidates else None
if key_predictor is None:
    raise RuntimeError('Could not find education_num in encoded feature names.')
sex_col = 'sex'
subgroup_results = []
for label, mask in [('Full', pd.Series(True, index=X_train.index)), ('Female', X_train[sex_col].astype(str).str.strip().str.lower().eq('female')), ('Male', X_train[sex_col].astype(str).str.strip().str.lower().eq('male'))]:
    idx = X_train.index[mask]
    fit = fit_glm(Xtr_df.loc[idx], y_train.loc[idx])
    subgroup_results.append({'group':label,'n':len(idx),'predictor':key_predictor,'coef':fit.params[key_predictor],'ci_low':fit.conf_int().loc[key_predictor,0],'ci_high':fit.conf_int().loc[key_predictor,1],'p_value':fit.pvalues[key_predictor]})
subgroup_table = pd.DataFrame(subgroup_results)
display(subgroup_table)


## Part 2.2 — Omitted-variable bias
Drop `education_num` as a plausible confounder of income relationships, then compare coefficients for at least four common predictors. Report percentage change as `100 * (reduced - full) / abs(full)`; this signed denominator convention makes the direction transparent even if a baseline coefficient is negative. Interpret large changes as sensitivity to specification, not automatic proof of causal confounding.

In [ ]:
reduced_features = [c for c in Xtr_df.columns if c != key_predictor]
reduced_glm = fit_glm(Xtr_df[reduced_features], y_train)
# Choose four predictors with largest absolute full-model coefficients, excluding intercept and dropped variable.
common = [c for c in Xtr_df.columns if c in reduced_glm.params.index and c != key_predictor]
common = sorted(common, key=lambda c: abs(full_glm.params[c]), reverse=True)[:8]
coef_rows=[]
for c in common:
    a=float(full_glm.params[c]); b=float(reduced_glm.params[c])
    pct = np.nan if abs(a)<1e-10 else 100*(b-a)/abs(a)
    coef_rows.append({'predictor':c,'full_coef':a,'reduced_coef':b,'change':b-a,'pct_change':pct})
omv_table=pd.DataFrame(coef_rows)
display(omv_table.head(8))


## Part 2.3 — Adversarial subset
Identify test cases where probability is at least 0.75 but actual outcome is 0, or probability at most 0.25 but actual outcome is 1. If there are more than 50, sample 50 with `random_state=728`; otherwise retain all. Demographics are compared against the full test set. A subset characteristic is selected from available attributes by comparing its frequency in the wrong-prediction subset to the test set, and its actual percentage is reported.

In [ ]:
print("Available model and prediction variables:")
for name in sorted(globals()):
    if any(word in name.lower() for word in
           ["model", "logit", "clf", "reg", "prob", "pred"]):
        obj = globals()[name]
        print(name, "->", type(obj).__name__)

In [ ]:
# Part 2.3 — Adversarial Subset Construction

import numpy as np
import pandas as pd

# Use the existing logistic regression probabilities
y_prob = np.asarray(logit_prob).ravel()
y_true = np.asarray(y_test).ravel()

# Find high-confidence incorrect predictions
adversarial_mask = (
    ((y_prob >= 0.75) & (y_true == 0)) |
    ((y_prob <= 0.25) & (y_true == 1))
)

# Count all high-confidence wrong predictions before any 50-row sampling.
adv_count_total = int(adversarial_mask.sum())
adv_pct_test = float(adversarial_mask.mean()) if len(y_true) else float('nan')

# Construct adversarial subset
adv_df = X_test.loc[adversarial_mask].copy()
adv_df["actual_outcome"] = y_true[adversarial_mask]
adv_df["predicted_probability"] = y_prob[adversarial_mask]

# Keep at most 50 rows, as required
if len(adv_df) > 50:
    adv_df = adv_df.sample(n=50, random_state=728)

print("High-confidence wrong predictions:", adv_count_total)
print("Percentage of full test set:", f"{adversarial_mask.mean():.2%}")
print("Rows selected for analysis:", len(adv_df))

# Compare demographics with the full test set
for col in ["age", "sex", "race", "education"]:
    if col in X_test.columns:
        print(f"\n--- {col}: adversarial subset ---")
        print(
            adv_df[col].value_counts(normalize=True)
            .mul(100).round(2)
        )

        print(f"\n--- {col}: full test set ---")
        print(
            X_test[col].value_counts(normalize=True)
            .mul(100).round(2)
        )

# Find the most overrepresented characteristic
trait_candidates = []
trait_col = trait_val = "no qualifying trait"
trait_adv_rate = trait_test_rate = 0.0

for col in [
    "sex", "race", "education", "marital_status",
    "occupation", "workclass", "age"
]:
    if col in adv_df.columns and len(adv_df) > 0:
        value = adv_df[col].astype(str).value_counts().index[0]

        subset_rate = (
            adv_df[col].astype(str) == value
        ).mean()

        test_rate = (
            X_test[col].astype(str) == value
        ).mean()

        trait_candidates.append(
            (subset_rate - test_rate, col, value,
             subset_rate, test_rate)
        )

if trait_candidates:
    (
        difference, trait_col, trait_val,
        trait_adv_rate, trait_test_rate
    ) = max(trait_candidates, key=lambda item: item[0])

    print("\nMost overrepresented characteristic:")
    print(f"{trait_col} = {trait_val}")
    print(f"Adversarial subset: {trait_adv_rate:.2%}")
    print(f"Full test set: {trait_test_rate:.2%}")
    print(f"Difference: {difference:.2%}")
else:
    print("\nNo adversarial cases or usable characteristics found.")

## Part 2.4 — Fairness trade-offs
Sensitive attribute: `sex` (Female vs Male). For each threshold, compute the required ratio metrics. To avoid undefined divisions, ratios with zero denominator are recorded as `NaN` and called out in interpretation. DP and PP are Group A / Group B; EO follows the assignment's minimum of TPR ratio and FPR ratio. Values above 1 are possible, so the chart axes are capped at 1.5 as required.

In [ ]:
sensitive = X_test['sex'].astype(str).str.strip().str.lower().to_numpy()
group_a = sensitive == 'female'
group_b = sensitive == 'male'
y_true = np.asarray(y_test, dtype=int)
def safe_ratio(a,b):
    return np.nan if b == 0 else a/b
def group_metrics(mask, pred):
    yt=y_true[mask]; yp=pred[mask]
    tp=np.sum((yt==1)&(yp==1)); fn=np.sum((yt==1)&(yp==0))
    fp=np.sum((yt==0)&(yp==1)); tn=np.sum((yt==0)&(yp==0))
    tpr=safe_ratio(tp,tp+fn); fpr=safe_ratio(fp,fp+tn)
    selection=safe_ratio(np.sum(yp==1),len(yp))
    ppv=safe_ratio(tp,tp+fp)
    return {'TPR':tpr,'FPR':fpr,'selection_rate':selection,'PPV':ppv}
thresholds=np.round(np.arange(.05,.951,.1),2)
fair_rows=[]
for t in thresholds:
    pred=(logit_prob>=t).astype(int)
    a=group_metrics(group_a,pred); b=group_metrics(group_b,pred)
    dp=safe_ratio(a['selection_rate'],b['selection_rate'])
    eo=min(safe_ratio(a['TPR'],b['TPR']),safe_ratio(a['FPR'],b['FPR'])) if not (np.isnan(safe_ratio(a['TPR'],b['TPR'])) or np.isnan(safe_ratio(a['FPR'],b['FPR']))) else np.nan
    pp=safe_ratio(a['PPV'],b['PPV'])
    fair_rows.append({'threshold':t,'DP':dp,'EO':eo,'PP':pp,'TPR_A':a['TPR'],'TPR_B':b['TPR'],'FPR_A':a['FPR'],'FPR_B':b['FPR'],'selection_A':a['selection_rate'],'selection_B':b['selection_rate'],'PPV_A':a['PPV'],'PPV_B':b['PPV'],'accuracy':accuracy_score(y_true,pred)})
fairness=pd.DataFrame(fair_rows)
display(fairness.round(4))
plt.figure(figsize=(8,6))
plt.scatter(fairness['DP'], fairness['EO'])
for _,r in fairness.iterrows():
    if pd.notna(r['DP']) and pd.notna(r['EO']): plt.annotate(f"{r['threshold']:.2f}",(r['DP'],r['EO']),xytext=(4,4),textcoords='offset points',fontsize=8)
plt.xlim(0,1.5); plt.ylim(0,1.5); plt.xlabel('Demographic Parity ratio (Female / Male)'); plt.ylabel('Equalized Odds ratio'); plt.title('Fairness trade-off across decision thresholds'); plt.grid(True, alpha=.25); plt.tight_layout()
chart_path=os.path.join(OUT_DIR,'fairness_tradeoff.png'); plt.savefig(chart_path,dpi=160); plt.show()


## Part 3 — Evidence-grounded descriptive answers
The next cell prepares data-backed drafts for the eight required answers. Review and personalize the prose before submission. The assignment expects approximately 250–300 words for each answer; the generated drafts prioritize truthful numerical grounding over padding. Expand each draft with interpretation after inspecting all tables and the fairness chart.

In [ ]:
# Build evidence-grounded answer notes from computed outputs.

import numpy as np
import pandas as pd

def get_var(name, default=None):
    return globals().get(name, default)

# Safely read existing computed outputs.
cm_agreement = get_var("cm_agreement", None)
omv_table = get_var("omv_table", pd.DataFrame())
fairness = get_var("fairness", pd.DataFrame())
adv_df = get_var("adv_df", pd.DataFrame())

ols_median = get_var("ols_median", float("nan"))
adv_pct_test = get_var("adv_pct_test", float("nan"))

# Subgroup results: use whichever format is available.
subtxt = "Subgroup regression results are unavailable."

for key in ["subg_results", "subgroup_results", "subgroup_table"]:
    obj = get_var(key)
    if isinstance(obj, pd.DataFrame) and not obj.empty:
        subtxt = obj.to_string(index=False)
        break
    elif isinstance(obj, list) and obj:
        subtxt = str(obj)
        break

# Identify the largest coefficient shift.
most_shift = None

if isinstance(omv_table, pd.DataFrame) and not omv_table.empty:
    if "pct_change" in omv_table.columns:
        idx = omv_table["pct_change"].abs().fillna(-1).idxmax()
        most_shift = omv_table.loc[idx].to_dict()

# Find fairness results if the expected columns exist.
best_dp = None
best_eo = None
bal = None

if isinstance(fairness, pd.DataFrame) and not fairness.empty:
    if {"DP", "EO", "threshold"}.issubset(fairness.columns):
        best_dp = fairness.loc[(fairness["DP"] - 1).abs().idxmin()]
        best_eo = fairness.loc[(fairness["EO"] - 1).abs().idxmin()]

        balance_score = (
            (fairness["DP"] - 1).abs()
            + (fairness["EO"] - 1).abs()
        )
        bal = fairness.loc[balance_score.idxmin()]

# Adversarial subset statistics.
adv_sampled = len(adv_df) if isinstance(adv_df, pd.DataFrame) else 0
adv_total = int(get_var("adv_count_total", adv_sampled))
adv_pct_test = float(get_var("adv_pct_test", adv_total / len(y_test) if len(y_test) else float("nan")))

answers = {}

answers[1] = (
    f"OLS predictions were split at their test-set median "
    f"({ols_median:.4f}), while logistic regression used a fixed "
    f"probability threshold."
)

answers[2] = (
    "The subgroup analysis compared education_num coefficients "
    f"across groups. Available results: {subtxt}"
)

if most_shift is not None:
    answers[3] = (
        "The omitted-variable stress test examines coefficient changes "
        f"after removing education_num. Largest observed shift: {most_shift}"
    )
else:
    answers[3] = (
        "The omitted-variable stress test examines coefficient changes "
        "after removing education_num. No usable shift table was found."
    )

answers[4] = (
    "A practical hidden-confound audit would combine domain-expert "
    "review with external validation on a different sample or a later "
    "time period."
)

answers[5] = (
    f"The model made {adv_total} high-confidence wrong predictions in the test set "
    f"({adv_pct_test:.2%}); {adv_sampled} rows were selected for subset analysis."
)

if best_dp is not None and best_eo is not None:
    answers[6] = (
        f"DP is closest to 1 at threshold {best_dp['threshold']:.2f}; "
        f"EO is closest to 1 at threshold {best_eo['threshold']:.2f}."
    )
else:
    answers[6] = (
        "Fairness threshold results are unavailable. Run the fairness "
        "sweep cell and check its output columns."
    )

if bal is not None:
    answers[7] = (
        f"The compromise threshold minimizing |DP-1| + |EO-1| is "
        f"{bal['threshold']:.2f}, with DP={bal['DP']:.4f} "
        f"and EO={bal['EO']:.4f}."
    )
else:
    answers[7] = (
        "A compromise threshold should be selected by minimizing "
        "|DP-1| + |EO-1| after the fairness sweep has run."
    )

answers[8] = (
    "Demographic parity and equalized odds represent different fairness "
    "goals. Improving one measure does not necessarily improve the other."
)

for k, v in answers.items():
    print("\n" + "=" * 50)
    print(f"ANSWER {k}")
    print("=" * 50)
    print(v)
    print()

> **Submission check:** The eight Part 3 responses currently generated below are evidence notes, not full 250–300-word answers. Expand each answer to the assignment length, explain the computed results in your own words, and verify the regenerated PDF before submission. The notebook does not fabricate missing results.


## Part 4 — Reflection (~500 words)
The cell below creates a reflection grounded in this run's computed values. Edit it to reflect your own judgment and what you learned from the subgroup table, omitted-variable test, and adversarial rows.

In [ ]:
reflection = f"""I would not recommend unconditional deployment of this income classifier. At most, I would support a limited, monitored pilot after a domain-specific harm assessment, independent validation, and a documented process for appeal. The reason is that an overall performance score cannot show whether the model's errors and benefits are distributed acceptably.

First, the choice of threshold is not a purely technical fact. OLS's test-set median was {ols_median:.4f}, while the logistic model used 0.50. These different cutoffs produced {mismatch:,} disagreements ({mismatch_pct:.2f}%). A policy-maker who sees only accuracy would not know which people change categories when the threshold changes, or whether the cost of a false positive differs from that of a false negative.

Second, subgroup effects can be hidden by an overall model. The education_num coefficient and confidence intervals differed across the full sample and the Female/Male models: {subtxt}. Even where a sign reversal is absent, different magnitudes can indicate that one average relationship is not a complete description. Deployment should therefore include subgroup-specific evaluation and careful communication that associations are not causal effects.

Third, fairness criteria can conflict. In the threshold sweep, the DP-nearest threshold was {best_dp['threshold']:.2f} (DP={best_dp['DP']:.3f}, EO={best_dp['EO']:.3f}, PP={best_dp['PP']:.3f}), while the EO-nearest threshold was {best_eo['threshold']:.2f} (DP={best_eo['DP']:.3f}, EO={best_eo['EO']:.3f}, PP={best_eo['PP']:.3f}). No threshold should be called universally fair without stating which harms and groups are prioritized.

Fourth, confident errors matter even when aggregate metrics look acceptable. The model made {adv_total:,} high-confidence wrong predictions ({adv_pct_test:.2%} of the test set), with {adv_sampled} rows selected for detailed analysis under the assignment's probability rules. These cases deserve human review and error analysis, especially if they cluster in particular demographic or work/education categories. Finally, omitted-variable testing showed that the coefficient estimates change when education_num is removed; the full-versus-reduced table should be read as evidence of specification sensitivity, not as proof of a causal mechanism.

My conclusion is that a single number is an inadequate deployment decision. Responsible use requires transparent thresholds, independent validation, subgroup monitoring, an appeals process, and a commitment to stop or revise the system if harms exceed pre-agreed limits."""
print(reflection)


## Export the final report PDF
This final cell creates a PDF containing preprocessing, Part 1 results, all computed stress-test tables, the fairness chart, eight answer drafts, and the reflection. Run the notebook from top to bottom first. The report is generated only after the real dataset has been loaded and all analyses have run.

In [ ]:
%pip install reportlab

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as RLImage, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import cm
from xml.sax.saxutils import escape

def ptxt(s): return escape(str(s)).replace('\n','<br/>')
report_path=os.path.join(OUT_DIR,'CSET485_Assignment1_Report.pdf')
styles=getSampleStyleSheet()
styles.add(ParagraphStyle(name='Small',parent=styles['BodyText'],fontSize=8,leading=10,spaceAfter=5))
story=[]
story += [Paragraph('CSET485 — AI and Society',styles['Title']),Paragraph('Assignment #1: Predictive Models, Bias and Fairness',styles['Heading2']),Paragraph('Roll number: S24CSEU0728 | Dataset: Adult Census Income | random_state=728',styles['BodyText']),Spacer(1,10)]
story += [Paragraph('Preprocessing',styles['Heading1']),Paragraph(ptxt(f'Dataset rows after target cleaning: {len(df)}. Train rows: {len(X_train)}; test rows: {len(X_test)}. Numeric columns: {len(num_cols)}; categorical columns: {len(cat_cols)}. Missing values were imputed using training-set medians for numeric fields and most-frequent values for categorical fields. Categorical fields were one-hot encoded; numeric features standardized. Transformations were fit on training data only. Numeric extremes were inspected and retained rather than automatically removed.'),styles['BodyText']),Spacer(1,8)]
story += [Paragraph('Part 1 — Model comparison',styles['Heading1']),Paragraph(ptxt(f'OLS median prediction: {ols_median:.6f}. Logistic threshold: 0.50. Mismatched cases: {mismatch}/{len(y_test)} ({mismatch_pct:.2f}%). Logistic accuracy: {accuracy_score(y_test,logit_flag):.4f}; ROC-AUC: {roc_auc_score(y_test,logit_prob):.4f}.'),styles['BodyText'])]
def add_df_table(dataframe, max_rows=30, font=7):
    d=dataframe.copy().head(max_rows).replace({np.nan:'NA'})
    rows=[list(map(str,d.columns))]+[[str(x)[:48] for x in row] for row in d.values.tolist()]
    t=Table(rows, repeatRows=1, hAlign='LEFT')
    t.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#dce6f1')),('GRID',(0,0),(-1,-1),.3,colors.grey),('FONTNAME',(0,0),(-1,0),'Helvetica-Bold'),('FONTSIZE',(0,0),(-1,-1),font),('VALIGN',(0,0),(-1,-1),'TOP'),('LEFTPADDING',(0,0),(-1,-1),3),('RIGHTPADDING',(0,0),(-1,-1),3)]))
    return t
story += [add_df_table(pd.DataFrame(cm_agreement,index=['OLS low','OLS high'],columns=['Logistic low','Logistic high']).reset_index(names='OLS vs Logistic')),Spacer(1,8)]
story += [Paragraph('Part 2.1 — Subgroup coefficient comparison',styles['Heading1']),add_df_table(subgroup_table.round(5)),Spacer(1,8)]
story += [Paragraph('Part 2.2 — Omitted-variable bias',styles['Heading1']),Paragraph('Variable dropped: education_num. Percentage change is 100 × (reduced − full) / |full|.',styles['Small']),add_df_table(omv_table.round(4),max_rows=8),Spacer(1,8)]
story += [Paragraph('Part 2.3 — Adversarial subset',styles['Heading1']),Paragraph(ptxt(f'High-confidence wrong predictions: {adv_total}/{len(X_test)} ({adv_pct_test:.2%} of test set); analyzed rows: {len(adv_df)}. The notebook prints the demographic distributions compared with the full test set. Most overrepresented modal trait: {trait_col}={trait_val}, subset share={trait_adv_rate*100:.2f}% vs test share={trait_test_rate*100:.2f}%.'),styles['BodyText']),Spacer(1,8)]
story += [Paragraph('Part 2.4 — Fairness trade-off',styles['Heading1']),add_df_table(fairness[['threshold','DP','EO','PP','accuracy']].round(4)),Spacer(1,8)]
if os.path.exists(chart_path): story.append(RLImage(chart_path,width=15*cm,height=11*cm))
story.append(PageBreak())
story.append(Paragraph('Part 3 — Descriptive answers (drafts to review)',styles['Heading1']))
for k,v in answers.items():
    story += [Paragraph(f'Question {k}',styles['Heading2']),Paragraph(ptxt(v),styles['Small']),Spacer(1,5)]
story += [PageBreak(),Paragraph('Part 4 — Reflection',styles['Heading1']),Paragraph(ptxt(reflection),styles['BodyText'])]
story += [Spacer(1,12),Paragraph('Reproducibility note: This report is generated by running the accompanying notebook from top to bottom. All reported numeric findings are computed from the downloaded Adult dataset using random_state=728.',styles['Small'])]
doc=SimpleDocTemplate(report_path,pagesize=A4,rightMargin=1.5*cm,leftMargin=1.5*cm,topMargin=1.5*cm,bottomMargin=1.5*cm)
doc.build(story)
print('Wrote report:',report_path)
